# 02 SFT、偏好对齐、DPO 和评测原理

目标：理解预训练、SFT、RLHF/DPO 的区别，并用小张量手算 DPO loss、KL 约束、评测指标和过拟合风险。


## 1. 安装依赖


In [ ]:
from pathlib import Path

requirements_path = Path("requirements.txt")
if not requirements_path.exists():
    requirements_path = Path("../requirements.txt")

%pip install -r {requirements_path}


## 2. 三类训练目标

```text
预训练: 给定海量文本，做 next-token prediction，学语言和世界知识。
SFT: 给定 instruction -> answer，仍然做 next-token prediction，但通常只监督 assistant answer。
偏好对齐: 给定 prompt、chosen answer、rejected answer，让模型更偏向 chosen。
```


In [ ]:
training_stages = [
    {
        "stage": "Pretraining",
        "data": "大量无标注文本",
        "objective": "next-token cross entropy",
        "learns": "语言、知识、基本推理模式",
        "risk": "不一定听指令，可能续写网页或复述训练格式",
    },
    {
        "stage": "SFT",
        "data": "人工或合成 instruction-answer",
        "objective": "assistant answer token cross entropy",
        "learns": "按指令回答、格式、任务风格",
        "risk": "学到平均答案，不一定符合人类偏好或安全边界",
    },
    {
        "stage": "RLHF / DPO",
        "data": "chosen/rejected 偏好对",
        "objective": "提高 chosen 相对 rejected 的概率优势",
        "learns": "偏好、安全、风格、拒答边界",
        "risk": "过度对齐会损害能力，偏好数据有偏会放大偏差",
    },
]

for row in training_stages:
    print("=" * 100)
    for key, value in row.items():
        print(f"{key:10s}: {value}")


## 3. 序列 log probability

偏好训练比较的不是单个 token，而是整个 answer 在给定 prompt 下的 log probability。长度不同的时候要注意 sum 和 average 的差异。


In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

batch_size = 3
chosen_len = 4
rejected_len = 4
vocab_size = 8

# 假设这些 logits 已经是 answer token 对应位置的模型输出。
chosen_logits = torch.randn(batch_size, chosen_len, vocab_size)
rejected_logits = torch.randn(batch_size, rejected_len, vocab_size)
chosen_labels = torch.tensor([
    [1, 2, 3, 4],
    [2, 2, 5, 6],
    [3, 1, 7, 4],
])
rejected_labels = torch.tensor([
    [1, 2, 6, 4],
    [2, 3, 5, 6],
    [3, 1, 2, 4],
])


def sequence_logprob(logits, labels, average=False):
    log_probs = F.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(dim=-1, index=labels.unsqueeze(-1)).squeeze(-1)
    if average:
        return token_log_probs.mean(dim=-1)
    return token_log_probs.sum(dim=-1)


chosen_lp = sequence_logprob(chosen_logits, chosen_labels)
rejected_lp = sequence_logprob(rejected_logits, rejected_labels)
print("chosen logprob :", chosen_lp)
print("rejected logprob:", rejected_lp)
print("chosen advantage:", chosen_lp - rejected_lp)


## 4. DPO loss 的核心直觉

DPO 比较 policy model 和 reference model 的相对偏好优势：

```text
logit = beta * [(policy_chosen - policy_rejected) - (ref_chosen - ref_rejected)]
loss = -log sigmoid(logit)
```

如果 policy 比 reference 更偏向 chosen，loss 会下降。


In [ ]:
def dpo_loss(policy_chosen_lp, policy_rejected_lp, ref_chosen_lp, ref_rejected_lp, beta=0.1):
    policy_margin = policy_chosen_lp - policy_rejected_lp
    ref_margin = ref_chosen_lp - ref_rejected_lp
    logits = beta * (policy_margin - ref_margin)
    losses = -F.logsigmoid(logits)
    rewards_chosen = beta * (policy_chosen_lp - ref_chosen_lp)
    rewards_rejected = beta * (policy_rejected_lp - ref_rejected_lp)
    return losses.mean(), {
        "policy_margin": policy_margin,
        "ref_margin": ref_margin,
        "dpo_logits": logits,
        "implicit_reward_chosen": rewards_chosen,
        "implicit_reward_rejected": rewards_rejected,
        "preference_accuracy": (policy_margin > ref_margin).float().mean(),
    }


ref_chosen_lp = chosen_lp - torch.tensor([0.2, -0.1, 0.4])
ref_rejected_lp = rejected_lp - torch.tensor([0.8, 0.2, -0.3])
loss, info = dpo_loss(chosen_lp, rejected_lp, ref_chosen_lp, ref_rejected_lp, beta=0.2)
print("dpo loss:", float(loss))
for key, value in info.items():
    print(key, value)


## 5. beta 控制偏好强度

`beta` 越大，越强地推动 policy 偏离 reference；太大会带来不稳定或能力损失风险。


In [ ]:
for beta in [0.05, 0.1, 0.5, 1.0]:
    loss, info = dpo_loss(chosen_lp, rejected_lp, ref_chosen_lp, ref_rejected_lp, beta=beta)
    print(f"beta={beta:<4} loss={float(loss):.4f} logits={info['dpo_logits'].tolist()}")


## 6. KL 约束的直觉

RLHF/PPO 常用 KL penalty 约束 policy 不要离 reference 太远。DPO 也显式使用 reference model 来控制偏移。


In [ ]:
policy_logits = torch.randn(2, 5)
reference_logits = policy_logits + torch.randn(2, 5) * 0.5
policy_log_probs = F.log_softmax(policy_logits, dim=-1)
reference_log_probs = F.log_softmax(reference_logits, dim=-1)
policy_probs = policy_log_probs.exp()

kl_policy_ref = (policy_probs * (policy_log_probs - reference_log_probs)).sum(dim=-1)
print("KL(policy || reference):", kl_policy_ref)
print("mean KL:", float(kl_policy_ref.mean()))


## 7. 评测：不要只看一个 demo

模型评测通常要分层：

- 语言建模：validation loss、perplexity。
- 任务能力：准确率、F1、EM、BLEU/ROUGE、pass@k、工具调用成功率。
- 偏好质量：win rate、人工 A/B、pairwise preference。
- 安全：拒答边界、越狱、敏感内容、隐私泄露。
- 线上：满意度、重试率、延迟、成本、错误率。


In [ ]:
# 一个小型分类评测例子：accuracy、precision、recall、F1。
y_true = torch.tensor([1, 0, 1, 1, 0, 0, 1, 0])
y_pred = torch.tensor([1, 0, 0, 1, 0, 1, 1, 0])

tp = int(((y_true == 1) & (y_pred == 1)).sum())
tn = int(((y_true == 0) & (y_pred == 0)).sum())
fp = int(((y_true == 0) & (y_pred == 1)).sum())
fn = int(((y_true == 1) & (y_pred == 0)).sum())

accuracy = (tp + tn) / len(y_true)
precision = tp / max(tp + fp, 1)
recall = tp / max(tp + fn, 1)
f1 = 2 * precision * recall / max(precision + recall, 1e-12)

print({"tp": tp, "tn": tn, "fp": fp, "fn": fn})
print(f"accuracy={accuracy:.3f} precision={precision:.3f} recall={recall:.3f} f1={f1:.3f}")


## 8. 过拟合和数据泄漏的玩具例子

如果训练集和测试集泄漏，指标会虚高。LLM benchmark contamination 本质上也是类似问题：模型可能见过题目或答案。


In [ ]:
train_questions = {
    "什么是 KV cache？": "保存 attention key/value 以复用历史上下文。",
    "TTFT 是什么？": "首 token 延迟。",
    "DPO 比较什么？": "chosen 和 rejected 的相对偏好。",
}

test_questions = [
    "TTFT 是什么？",
    "DPO 比较什么？",
    "TPOT 是什么？",
]

leaked = [q for q in test_questions if q in train_questions]
print("test size:", len(test_questions))
print("leaked questions:", leaked)
print("leak ratio:", len(leaked) / len(test_questions))


## 面试总结

- 预训练、SFT、偏好对齐不是三套完全不同模型，本质都是在调整 token 概率分布，只是数据和目标不同。
- SFT 用 teacher forcing 学标准答案；偏好对齐比较 chosen/rejected，让模型更偏向人类喜欢的回答。
- DPO 直接用 policy/reference 的 log probability 差异构造偏好 loss，不需要显式训练 reward model 和跑 PPO。
- KL/reference 约束是为了防止模型为了迎合偏好数据而偏离基础能力太远。
- 评测要覆盖能力、偏好、安全、鲁棒性和线上指标；单条 demo 不能证明模型好。
- 数据泄漏和 benchmark contamination 会让离线分数虚高，真实上线前要做隔离测试和人工抽检。
